# Final Project Roster Management Program
Conor Murray

In [2]:
import math
semester1_roster = []
semester2_roster = []

In [3]:
def get_filename(prompt_text):
    while True:
        filename = input(prompt_text)
        try:
            file_object = open(filename, "r")
            return file_object
        except FileNotFoundError:
            print(f"Sorry, could not find a file named '{filename}'. Please try again.")

In [4]:
def parse_name(full_name):
    parts = full_name.strip().split()
    first_name = parts[0]
    last_name = parts[-1]
    middle_name = " ".join(parts[1:-1])
    return first_name, middle_name, last_name

In [5]:
def parse_score(score_text):
    score_text = score_text.strip()
    if score_text == "":
        return None
    return float(score_text)

In [6]:
def load_roster(file_object):
    lines = file_object.readlines()
    file_object.close()
    roster = []
    for line in lines:
        line = line.strip()
        if line == "":
            continue
        parts = line.split(";")
        full_name = parts[0].strip()
        exam1 = parse_score(parts[1])
        exam2 = parse_score(parts[2])
        first_name, middle_name, last_name = parse_name(full_name)
        roster.append([full_name, first_name, middle_name, last_name, exam1, exam2])
    return roster

In [7]:
def name_key(first_name, last_name):
    return first_name.strip().lower() + " " + last_name.strip().lower()

In [8]:
def compute_stats(score_list):
    count = len(score_list)
    if count == 0:
        return 0, 0, 0
    total = 0
    for score in score_list:
        total = total + score
    average = total / count
    if count > 1:
        squared_diff_total = 0
        for score in score_list:
            squared_diff_total = squared_diff_total + (score - average) ** 2
        variance = squared_diff_total / (count - 1)
        std_dev = math.sqrt(variance)
    else:
        std_dev = 0
    return count, average, std_dev

In [9]:
def find_dropped():
    semester2_keys = []
    for student in semester2_roster:
        semester2_keys.append(name_key(student[1], student[3]))
    dropped_names = []
    for student in semester1_roster:
        key = name_key(student[1], student[3])
        if key not in semester2_keys:
            dropped_names.append(student[0])
    return dropped_names

In [10]:
def summary_statistics(semesterID):
    if semesterID == 1:
        roster = semester1_roster
    else:
        roster = semester2_roster
    exam1_scores = []
    exam2_scores = []
    for student in roster:
        if student[4] is not None:
            exam1_scores.append(student[4])
        if student[5] is not None:
            exam2_scores.append(student[5])
    exam1_count, exam1_avg, exam1_std = compute_stats(exam1_scores)
    exam2_count, exam2_avg, exam2_std = compute_stats(exam2_scores)
    print(f"Semester {semesterID} Summary Statistics")
    print(f"  Exam 1: {exam1_count} students took it, average = {round(exam1_avg, 2)}, standard deviation = {round(exam1_std, 2)}")
    print(f"  Exam 2: {exam2_count} students took it, average = {round(exam2_avg, 2)}, standard deviation = {round(exam2_std, 2)}")

In [11]:
def student_performance(studentName):
    parts = studentName.strip().split()
    if len(parts) < 2:
        print("Please enter both a first and last name.")
        return
    first_name = parts[0]
    last_name = parts[-1]
    key = name_key(first_name, last_name)
    semester1_match = None
    for student in semester1_roster:
        if name_key(student[1], student[3]) == key:
            semester1_match = student
    semester2_match = None
    for student in semester2_roster:
        if name_key(student[1], student[3]) == key:
            semester2_match = student
    if semester1_match is None and semester2_match is None:
        print("No student found with that name.")
        return
    print(f"Performance for {studentName.title()}:")
    if semester1_match is None:
        print("  Semester 1: Not enrolled")
    else:
        exam1_text = "Not taken" if semester1_match[4] is None else semester1_match[4]
        exam2_text = "Not taken" if semester1_match[5] is None else semester1_match[5]
        print(f"  Semester 1: Exam 1 = {exam1_text}, Exam 2 = {exam2_text}")

    if semester2_match is None:
        print("  Semester 2: Not enrolled")
    else:
        exam1_text = "Not taken" if semester2_match[4] is None else semester2_match[4]
        exam2_text = "Not taken" if semester2_match[5] is None else semester2_match[5]
        print(f"  Semester 2: Exam 1 = {exam1_text}, Exam 2 = {exam2_text}")

In [12]:
def cutoff_report(cutoffScore, semesterID, examID):
    if semesterID == 1:
        roster = semester1_roster
    else:
        roster = semester2_roster

    if examID == 1:
        exam_index = 4
    else:
        exam_index = 5

    below_cutoff = []
    for student in roster:
        score = student[exam_index]
        if score is not None and score < cutoffScore:
            below_cutoff.append(student[0])
    return below_cutoff

In [13]:
def main():
    global semester1_roster, semester2_roster
    print("Welcome to the Roster Management Program")
    file_object1 = get_filename("Enter the filename for Semester 1: ")
    semester1_roster = load_roster(file_object1)
    file_object2 = get_filename("Enter the filename for Semester 2: ")
    semester2_roster = load_roster(file_object2)
    keep_going = True
    while keep_going:
        print()
        print("What would you like to do?")
        print("1. Find students who dropped the course")
        print("2. View summary statistics for a semester")
        print("3. View a student's performance")
        print("4. View a cutoff report")
        choice = input("Enter your choice (1-4): ")
        if choice == "1":
            dropped_names = find_dropped()
            if len(dropped_names) == 0:
                print("No students dropped the course.")
            else:
                print("The following students dropped the course:")
                for name in dropped_names:
                    print(f"  {name}")
        elif choice == "2":
            semester_text = input("Which semester? (1 or 2): ")
            summary_statistics(int(semester_text))
        elif choice == "3":
            student_name = input("Enter the student's first and last name: ")
            student_performance(student_name)
        elif choice == "4":
            cutoff_text = input("Enter the cutoff score: ")
            semester_text = input("Which semester? (1 or 2): ")
            exam_text = input("Which exam? (1 or 2): ")
            below_cutoff = cutoff_report(float(cutoff_text), int(semester_text), int(exam_text))
            if len(below_cutoff) == 0:
                print("No students scored below the cutoff.")
            else:
                print(f"Students who scored below {cutoff_text}:")
                for name in below_cutoff:
                    print(f"  {name}")
        else:
            print("That is not a valid choice. Please enter 1, 2, 3, or 4.")
        again = input("Run again? (Y / N): ")
        if again.strip().upper() != "Y":
            keep_going = False
    print("Thanks for using the Roster Management Program.")

In [17]:
main()

Welcome to the Roster Management Program


Enter the filename for Semester 1:  semester1.txt
Enter the filename for Semester 2:  semester2.txt



What would you like to do?
1. Find students who dropped the course
2. View summary statistics for a semester
3. View a student's performance
4. View a cutoff report


Enter your choice (1-4):  1


The following students dropped the course:
  Robert Paul Brown
  Emily R. Clark


...


What would you like to do?
1. Find students who dropped the course
2. View summary statistics for a semester
3. View a student's performance
4. View a cutoff report


Enter your choice (1-4):  4
Enter the cutoff score:  96
Which semester? (1 or 2):  1
Which exam? (1 or 2):  1


Students who scored below 96:
  John A. Smith
  Jane Beth Doe
  Michael J. Lee
  Robert Paul Brown
  Emily R. Clark


Run again? (Y / N):  n


Thanks for using the Roster Management Program.
